# Lab 9: Model Evaluation & Metrics

**Author:** Arslan Hashmi  
**Track:** AI/ML Internship Program | Week 2  
**Institution:** National University of Technology (NUTECH)

---

Welcome to Lab 9! So far you've measured your classifiers using accuracy alone. In this lab you'll go deeper: confusion matrices, precision, recall, F1-score, ROC curves, and cross-validation.

**Topics covered:**
- Confusion Matrix
- Precision, Recall & F1-Score
- ROC Curve & AUC
- Cross-Validation
- Threshold tuning

This notebook focuses on the practical, hands-on workflow.

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.

## Getting the Dataset

This lab uses the Titanic dataset. It loads directly from a URL — no download or upload needed.

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.shape)
print(df["Survived"].value_counts())   # notice the class imbalance
df.head()

## Quick Preparation

A short cleanup before modeling: fill missing `Age` with the median, encode `Sex` as 0/1, and select a handful of relevant features. (Full data cleaning was covered in Lab 3 — here we keep it minimal since the focus is evaluation.)

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
X = df[features]
y = df["Survived"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model trained. Test accuracy:", model.score(X_test, y_test))

## Example 1: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
print(cm)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Did Not Survive", "Survived"],
            yticklabels=["Did Not Survive", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Example 2: Precision, Recall & F1-Score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

print("\nFull classification report:")
print(classification_report(y_test, y_pred))

## Example 3: ROC Curve & AUC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_probs = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc = roc_auc_score(y_test, y_probs)

plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Example 4: Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("Accuracy for each of the 5 folds:", cv_scores)
print("Average accuracy:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

## Practice Exercise 1: Evaluate a Different Model

**Exercise:** Train a Decision Tree classifier (`max_depth=4`) on the same `X_train`, `y_train`. Compute and print its precision, recall, and F1-score on the test set. How do these compare to Logistic Regression?

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = None
precision_tree = None
recall_tree = None
f1_tree = None

### YOUR CODE HERE ###
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)
### END ###

print("Precision:", round(precision_tree, 3))
print("Recall:", round(recall_tree, 3))
print("F1-Score:", round(f1_tree, 3))
print("\nCompared to Logistic Regression — Decision Tree often shows different precision/recall trade-offs.")

## Practice Exercise 2: Changing the Decision Threshold

By default, a probability >= 0.5 is classified as the positive class. Lowering this threshold increases recall (catches more positives) but usually decreases precision.

**Exercise:** Using `y_probs` from Example 3, create new predictions using a threshold of 0.3 instead of 0.5, then compute the new precision and recall. Compare them to the original (threshold 0.5) values from Example 2.

In [ ]:
y_pred_threshold_03 = None
precision_03 = None
recall_03 = None

### YOUR CODE HERE ###
y_pred_threshold_03 = (y_probs >= 0.3).astype(int)
precision_03 = precision_score(y_test, y_pred_threshold_03)
recall_03 = recall_score(y_test, y_pred_threshold_03)
### END ###

print("Precision (threshold=0.3):", round(precision_03, 3))
print("Recall (threshold=0.3):", round(recall_03, 3))
print("\n→ Lower threshold (0.3) increases Recall but usually decreases Precision.")

## Practice Exercise 3: Cross-Validation with a Different Model

**Exercise:** Run 5-fold cross-validation on a Decision Tree (`max_depth=4`) instead of Logistic Regression, using the same `X` and `y`. Print the average accuracy and compare it to the Logistic Regression cross-validation score from Example 4.

In [ ]:
### YOUR CODE HERE ###
tree_cv = DecisionTreeClassifier(max_depth=4, random_state=42)
cv_scores_tree = cross_val_score(tree_cv, X, y, cv=5, scoring="accuracy")

print("Accuracy for each of the 5 folds:", cv_scores_tree)
print("Average accuracy (Decision Tree):", round(cv_scores_tree.mean(), 4))
print("Standard deviation:", round(cv_scores_tree.std(), 4))
print("\nCompare with Logistic Regression CV average from Example 4.")
### END ###

## Lab Tasks

Complete the following in this notebook, below this cell.

1. **Full Evaluation**: For your Logistic Regression model, print the confusion matrix, precision, recall, F1-score, and AUC in one organized cell.
2. **Model Comparison**: Train Logistic Regression, KNN (k=5), and a Decision Tree (max_depth=4) on this dataset. Create a summary table comparing their precision, recall, and F1-score.
3. **Threshold Analysis**: Try 3 different thresholds (e.g., 0.3, 0.5, 0.7) on your Logistic Regression model's probabilities. For each, print precision and recall, and explain the trade-off you observe.
4. **Cross-Validation Comparison**: Run 5-fold cross-validation for all three models from Task 2, and report which model has the most consistent (lowest standard deviation) performance across folds.
5. **Reflection**: In a markdown cell, write 3-4 sentences on which metric (precision, recall, or F1) you think matters most for predicting Titanic survival, and why.

### Task 1 — Full Evaluation (Logistic Regression)

In [ ]:
# Task 1: Full Evaluation

print("=== Logistic Regression — Full Evaluation ===\n")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print()

# Metrics
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_probs)

print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"AUC       : {auc_score:.4f}")

### Task 2 — Model Comparison

In [ ]:
# Task 2: Model Comparison
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42)
}

results = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    results.append({
        "Model": name,
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1-Score": round(f1_score(y_test, preds), 4)
    })

comparison_df = pd.DataFrame(results)
print("=== Model Comparison Table ===")
print(comparison_df.to_string(index=False))

### Task 3 — Threshold Analysis

In [ ]:
# Task 3: Threshold Analysis

print("=== Threshold Analysis (Logistic Regression) ===\n")

thresholds_to_try = [0.3, 0.5, 0.7]

for thresh in thresholds_to_try:
    preds_t = (y_probs >= thresh).astype(int)
    p = precision_score(y_test, preds_t)
    r = recall_score(y_test, preds_t)
    print(f"Threshold = {thresh}")
    print(f"  Precision: {p:.4f}")
    print(f"  Recall   : {r:.4f}\n")

print("Observation:")
print("- Lower threshold (0.3) → higher Recall, lower Precision (more survivors predicted, more false positives)")
print("- Higher threshold (0.7) → higher Precision, lower Recall (only high-confidence survivors, more missed cases)")
print("- Threshold 0.5 is the balanced default.")

### Task 4 — Cross-Validation Comparison

In [ ]:
# Task 4: Cross-Validation Comparison

print("=== 5-Fold Cross-Validation Comparison ===\n")

cv_results = []

for name, clf in models.items():
    scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
    cv_results.append({
        "Model": name,
        "Mean Accuracy": round(scores.mean(), 4),
        "Std Dev": round(scores.std(), 4)
    })

cv_df = pd.DataFrame(cv_results)
print(cv_df.to_string(index=False))

# Find most consistent (lowest std)
most_consistent = cv_df.loc[cv_df["Std Dev"].idxmin()]
print(f"\nMost consistent model: {most_consistent['Model']} (lowest Std Dev = {most_consistent['Std Dev']})")

### Task 5 — Reflection

For predicting Titanic survival, **Recall** is the most important metric. Missing a passenger who actually survived (false negative) means we fail to identify someone who should have been saved — a costly error in a life-or-death context. While precision also matters (we don't want to over-predict survival), the cost of a false negative is higher than a false positive. F1-score is useful as a balanced summary, but when forced to choose one priority, maximizing recall better aligns with the real-world goal of correctly identifying as many survivors as possible.

---

**Lab completed by Arslan Hashmi**  
AI/ML Internship Program | Week 2  
National University of Technology (NUTECH)